# 02 — Phase 2 QLoRA training (Colab / cloud GPU)

Runs on **CUDA** with bitsandbytes 4-bit nf4 quantization. Don't run this notebook on Apple Silicon — the bnb path will fail.

Tracks issue: https://github.com/Charlemagne-Labs/gateguard-suite/issues/73 (Phase 2)

## 1. Install

On Colab, mount the repo (clone or upload) and install with the `[train]` extra:

```bash
!git clone https://github.com/Charlemagne-Labs/g4h.git
%cd g4h
!pip install -e ".[train]"
```

In [ ]:
import torch
import transformers

assert torch.cuda.is_available(), "This notebook requires CUDA. Run 01_smoke_test.ipynb on Mac instead."

print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"CUDA device:  {torch.cuda.get_device_name(0)}")
print(f"CUDA memory:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. QLoRA config (per issue #73)

In [ ]:
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

MODEL_ID = "google/gemma-4-e4b"  # TODO: verify against the Gemma 4 release

# LoRA hyperparameters (per issue):
#   rank 8-16, alpha 16-32, lr 1e-4, 1-3 epochs, batch 4-8
LORA_R = 8
LORA_ALPHA = 16
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj"]  # confirm in 01_smoke_test step 6

LR = 1e-4
EPOCHS = 2
BATCH = 4
MAX_LENGTH = 512

## 3. Train

Once `src/model.py` and `src/train.py` are filled in, this cell becomes a single call:

```python
from src.train import run_training

run_training(
    model_id=MODEL_ID,
    bnb_config=bnb_config,
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_target_modules=LORA_TARGET_MODULES,
    lr=LR, epochs=EPOCHS, batch=BATCH, max_length=MAX_LENGTH,
    train_csv="data/train.csv", eval_csv="data/eval.csv",
    out_dir="runs/gemma4-e4b-cls/",
)
```

Until then this cell is a placeholder.

In [ ]:
# TODO(phase-2): wire up src.train.run_training
raise NotImplementedError("Fill in src/model.py and src/train.py first (Phase 2)")

## 4. Save artifacts to a tarball

Output shape (matches gateguard-suite's `inference_config.json` schema so artifacts are interchangeable):

```
runs/gemma4-e4b-cls/
├── adapter/                 # PEFT save_pretrained
├── classifier_head.pt
├── inference_config.json    # max_length, label2id, is_merged, hidden_size
└── label_map.json
```

Then `tar -czf teacher_model.tar.gz runs/gemma4-e4b-cls/` so it can be uploaded to the `charley-s3-teacher-model` bucket and consumed by `Charlemagne-Labs/test_suite/phase2`.